In [1]:
# Run using ms-toolsai.jupyter for VSCode

In [ ]:
# Install dependencies
%pip install anthropic python-dotenv

In [ ]:
#  Load env vars
from dotenv import load_dotenv
# import os

load_dotenv()
# print(os.environ['ANTHROPIC_API_KEY'])

In [4]:
# Create API client
from anthropic import Anthropic

client = Anthropic()
model = "claude-haiku-4-5"

In [5]:
# Helper functions to preserve history
def add_user_message(messages, text):
  user_message = {"role": "user", "content": text}
  messages.append(user_message)

def add_assistant_message(messages, text):
  assistant_message = {"role": "assistant", "content": text}
  messages.append(assistant_message)

def chat(messages, system=None, temperature=0.5, stop_sequences=[]):
  params = {
    "model": model,
    "max_tokens": 1000,
    "messages": messages,
    "temperature": temperature,
    "stop_sequences": stop_sequences,
  }
  
  if system:
    params["system"] = system
  
  message = client.messages.create(**params)
  return message.content[0].text

In [ ]:
import json

def generate_dataset(n_of_test_cases=3):
  """"Generates a dataset for a prompt evaluation"""

  prompt = f"""
Generate an evaluation dataset for a prompt evaluation. The dataset will be used to evaluate prompts that generate Python, JSON, or Regex specifically for AWS-related tasks. Generate an array of JSON objects, each representing task that requires Python, JSON, or a Regex to complete.

Example output:
```json
[
  {
    "task": "Description of task",
    "format": "json" or "python" or "regex",
    "solution_criteria": "Key criteria for the solution"
  },
  ...additional
]
```

* Focus on tasks that can be solved by writing a single Python function, a single JSON object, or a single regex
* Focus on tasks that do not require writing much code

Please generate {n_of_test_cases} objects.
"""

  messages = []
  add_user_message(messages, prompt)
  add_assistant_message(messages, "```json")
  text = chat(messages, stop_sequences=["```"])
  return json.loads(text)

In [7]:
dataset = generate_dataset()

with open('dataset.json', 'w') as f:
  json.dump(dataset, f, indent=2)

In [ ]:
def run_prompt(test_case):
    """Merges the prompt and test case input, then returns the result"""
    
    # THIS is the meat of it. The actual PROMPT we want to evaluate.
    prompt = f"""
Please solve the following task:

{test_case["task"]}

* Respond only with Python, JSON or a plain Regex
* Do not add any comments or commentary explanation
"""
    
    messages = []
    add_user_message(messages, prompt)
    add_assistant_message(messages, "```code") # control output format
    output = chat(messages, stop_sequences=["```"])
    return output

In [9]:
# Function to grade a test case + output using a model
def grade_by_model(test_case, output):
  eval_prompt = f"""
You are an expert AWS code reviewer. Your task is to evaluate the following AI-generated solution.

Original Task:
<task>
{test_case["task"]}
</task>

Solution to Evaluate:
<solution>
{output}
</solution>

Criteria you should use to evaluate the solution:
<criteria>
{test_case["solution_criteria"]}
</criteria>

Output Format
Provide your evaluation as a structured JSON object with the following fields, in this specific order:
- "strengths": An array of 1-3 key strengths
- "weaknesses": An array of 1-3 key areas for improvement
- "reasoning": A concise explanation of your overall assessment
- "score": A number between 1-10

Respond with JSON. Keep your response concise and direct.
Example response shape:
{{
  "strengths": string[],
  "weaknesses": string[],
  "reasoning": string,
  "score": number
}}
"""

  messages = []
  add_user_message(messages, eval_prompt)
  add_assistant_message(messages, "```json")
  eval_text = chat(messages, stop_sequences=["```"])
  return json.loads(eval_text)

In [10]:
# Functions to validate the output structure
import re
import ast

def validate_json(text):
    try:
        json.loads(text.strip())
        return 10
    except json.JSONDecodeError:
        return 0


def validate_python(text):
    try:
        ast.parse(text.strip())
        return 10
    except SyntaxError:
        return 0


def validate_regex(text):
    try:
        re.compile(text.strip())
        return 10
    except re.error:
        return 0


def grade_syntax(test_case, response):
    format = test_case["format"]
    if format == "json":
        return validate_json(response)
    elif format == "python":
        return validate_python(response)
    else:
        return validate_regex(response)


In [ ]:
def run_test_case(test_case):
    """Calls run_prompt, then grades the result"""
    
    output = run_prompt(test_case)
    model_grade = grade_by_model(test_case, output)
    
    reasoning = model_grade["reasoning"]

    model_score = model_grade["score"]
    syntax_score = grade_syntax(test_case, output)

    score = (model_score + syntax_score) / 2

    return {
        "test_case": test_case,
        "output": output,
        "score": score,
        "reasoning": reasoning
    }

In [12]:
from statistics import mean

def run_eval(dataset):
    """Loads the dataset and calls run_test_case with each case"""
    
    results = []
    
    for test_case in dataset:
        result = run_test_case(test_case)
        results.append(result)
    
    average_score = mean([result["score"] for result in results])
    print(f"Average score: {average_score}")

    return results

In [13]:
with open("dataset.json", "r") as f:
    dataset = json.load(f)

results = run_eval(dataset)

Average score: 8.166666666666666


In [14]:
print(json.dumps(results, indent=2))

[
  {
    "test_case": {
      "task": "Parse an AWS S3 bucket name from a full S3 URI (e.g., 's3://my-bucket-name/path/to/file.txt') and extract just the bucket name",
      "format": "regex",
      "solution_criteria": "The regex should correctly extract 'my-bucket-name' from various S3 URI formats and handle bucket names with hyphens and numbers"
    },
    "output": "\nimport re\n\ndef extract_s3_bucket(uri):\n    match = re.match(r's3://([^/]+)', uri)\n    return match.group(1) if match else None\n",
    "score": 8.5,
    "reasoning": "The solution correctly solves the core requirement of extracting bucket names from standard S3 URIs using an appropriate regex pattern. The `[^/]+` character class properly captures bucket names containing hyphens and numbers. However, it lacks defensive programming practices such as input validation and comprehensive error handling. The implementation would benefit from type hints, documentation, and validation against actual AWS S3 bucket naming c